# 다음 투구 제구 성공 확률 — 모델 대시보드

합성 데이터 생성부터 시간 기반 검증, 모델 비교, 최종 예측까지 한 번에 실행합니다.

> 현재 결과는 `provisional-v1` 임시 스키마와 합성 데이터 기준입니다. 공식 데이터 성능이 아닙니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "provisional_schema.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
OUTPUT_DIR = PROJECT_ROOT / "data" / "generated"

print(f"Project root: {PROJECT_ROOT}")
print(f"Output: {OUTPUT_DIR}")

## 1. 전체 파이프라인 실행

2019~2025 합성 데이터를 만들고 2023·2024 시간 Fold로 후보 모델을 비교합니다. 평균 Log Loss가 가장 낮은 모델을 전체 학습 데이터로 재학습해 2025 제출 파일을 생성합니다.

In [ ]:
from baseball_platform.pipeline import run

result = run(OUTPUT_DIR)
result

## 2. 모델 리더보드

Log Loss와 Brier Score는 낮을수록, ROC-AUC는 높을수록 좋습니다. 공식 평가 지표가 공개되면 순위 기준을 교체해야 합니다.

In [ ]:
import pandas as pd
from IPython.display import display

leaderboard = pd.read_csv(result["leaderboard"])
display(
    leaderboard.style
    .format({
        "mean_log_loss": "{:.6f}",
        "std_log_loss": "{:.6f}",
        "mean_brier_score": "{:.6f}",
        "mean_roc_auc": "{:.6f}",
    })
    .background_gradient(subset=["mean_log_loss"], cmap="RdYlGn_r")
    .background_gradient(subset=["mean_roc_auc"], cmap="RdYlGn")
)


## 3. 시즌 Fold별 상세 결과

평균뿐 아니라 특정 시즌에서 성능이 크게 나빠지는지도 확인합니다.

In [ ]:
fold_results = pd.read_csv(result["fold_results"])
display(
    fold_results.sort_values(["validation_season", "log_loss"])
    .style.format({
        "log_loss": "{:.6f}",
        "brier_score": "{:.6f}",
        "roc_auc": "{:.6f}",
    })
)


## 4. 성능 비교 대시보드

In [ ]:
from IPython.display import Image, display

display(Image(filename=result["dashboard"]))

## 5. 2025 예측 결과 확인

확률 범위, 결측치, 행 수와 일부 예측을 확인합니다.

In [ ]:
submission = pd.read_csv(result["submission"])
probability = submission["control_success_probability"]

summary = pd.DataFrame({
    "rows": [len(submission)],
    "missing": [int(probability.isna().sum())],
    "minimum": [probability.min()],
    "mean": [probability.mean()],
    "maximum": [probability.max()],
})
display(summary.style.format({"minimum": "{:.4f}", "mean": "{:.4f}", "maximum": "{:.4f}"}))
display(submission.head(10))

## 6. 예측 확률 분포

모든 확률이 평균 주변에 몰리거나 0과 1에 지나치게 집중되는지 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

ax = probability.plot.hist(bins=20, figsize=(9, 4), color="#2563EB", edgecolor="white")
ax.set_title("2025 Synthetic Prediction Probability Distribution")
ax.set_xlabel("Control success probability")
ax.set_ylabel("Pitch count")
ax.grid(axis="y", alpha=0.25)
plt.show()

## 다음 확장

공식 데이터가 제공되면 이 Notebook에 Target 분포, 결측치, Calibration curve, 투수·상황별 성능, Feature importance 및 오류 분석을 추가합니다.